In [2]:
ACTUAL_NETWORK_BUSES = [
  'AL00',
  'AT00',
  'BA00',
  'BE00',
  'BG00',
  'CH00',
  'CY00',
  'CZ00', 
  'DE00',
  'DKE1',
  'DKW1',
  'EE00',
  'ES00',
  'FI00',
  'FR00',
  'FR15',
  'GB00',
  'GBNI',
  'GR00',
  'GR03',
  'HR00',
  'HU00',
  'IE00',
  'ITCA',
  'ITCN',
  'ITCS',
  'ITN1',
  'ITS1',
  'ITSA',
  'ITSI',
  'LT00',
  'LUB1',
  'LUF1',
  'LUG1',
  'LUV1',
  'LV00',
  'ME00',
  'MK00',
  'MT00',
  'NL00',
  'NOM1',
  'NON1',
  'NOS0',
  'PL00',
  'PT00',
  'RO00',
  'RS00',
  'SE01',
  'SE02',
  'SE03',
  'SE04',
  'SI00',
  'SK00',
  'ITCO',
  'ITVI'
]

In [3]:
print(len(ACTUAL_NETWORK_BUSES))

55


## Check Column Alignment — PECD profiles vs. Network Buses

Before using any PECD profile, we need to verify that the **column names match the bus codes** in `buses.csv`.

We check three files:
- `pecd_data_Wind_Onshore_2030.csv`
- `pecd_data_LFSolarPVUtility_2030.csv`
- `pecd_data_LFSolarPVRooftop_2030.csv`

In [4]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/open-tyndp")

actual_buses = set(ACTUAL_NETWORK_BUSES)

# Load buses.csv
buses_csv = pd.read_csv(DATA_DIR / "buses.csv")
buses_csv_ids = set(buses_csv['bus_id'])

print("=== buses.csv vs ACUTAL_NETWORK_BUSES ===")
only_in_csv    = sorted(buses_csv_ids - actual_buses)
only_in_actual = sorted(actual_buses  - buses_csv_ids)
print(f"  In buses.csv only   : {only_in_csv    if only_in_csv    else 'none'}")
print(f"  In ACTUAL only      : {only_in_actual if only_in_actual else 'none'}")
print(f"  Matched             : {len(buses_csv_ids & actual_buses)}")

# Load PECD profiles
profiles = {
    'Wind Onshore' : pd.read_csv(DATA_DIR / "pecd_data_Wind_Onshore_2030.csv",     index_col=0, parse_dates=True),
    'Solar Utility': pd.read_csv(DATA_DIR / "pecd_data_LFSolarPVUtility_2030.csv", index_col=0, parse_dates=True),
    'Solar Rooftop': pd.read_csv(DATA_DIR / "pecd_data_LFSolarPVRooftop_2030.csv", index_col=0, parse_dates=True),
}

print("\n=== PECD profiles vs ACUTAL_NETWORK_BUSES ===")
for name, df in profiles.items():
    cols = set(df.columns)
    only_profile = sorted(cols - actual_buses)
    only_actual  = sorted(actual_buses - cols)
    print(f"\n  {name}")
    print(f"    In profile, NOT in ACTUAL : {only_profile if only_profile else 'none'}")
    print(f"    In ACTUAL,  NOT in profile: {only_actual  if only_actual  else 'none'}")
    print(f"    Matched: {len(cols & actual_buses)}")


=== buses.csv vs ACUTAL_NETWORK_BUSES ===
  In buses.csv only   : none
  In ACTUAL only      : none
  Matched             : 55

=== PECD profiles vs ACUTAL_NETWORK_BUSES ===

  Wind Onshore
    In profile, NOT in ACTUAL : none
    In ACTUAL,  NOT in profile: none
    Matched: 55

  Solar Utility
    In profile, NOT in ACTUAL : none
    In ACTUAL,  NOT in profile: none
    Matched: 55

  Solar Rooftop
    In profile, NOT in ACTUAL : none
    In ACTUAL,  NOT in profile: none
    Matched: 55

=== PECD profiles vs ACUTAL_NETWORK_BUSES ===

  Wind Onshore
    In profile, NOT in ACTUAL : none
    In ACTUAL,  NOT in profile: none
    Matched: 55

  Solar Utility
    In profile, NOT in ACTUAL : none
    In ACTUAL,  NOT in profile: none
    Matched: 55

  Solar Rooftop
    In profile, NOT in ACTUAL : none
    In ACTUAL,  NOT in profile: none
    Matched: 55


## Offshore Zone → Home Node Mapping

`NODE.csv` contains the authoritative mapping: `OFFSHORE_NODE → HOME_NODE`.

The PECD offshore CSV uses `GB*` zone names while NODE.csv uses `UK*` — we handle that rename,
and also map `UK00 → GB00` / `UKNI → GBNI` to match our network bus codes.


In [5]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/open-tyndp")

actual_buses = set(ACTUAL_NETWORK_BUSES)

# Load NODE.csv
node_df = pd.read_csv(DATA_DIR / "NODE.csv", sep=";")

# TYNDP NODE.csv uses UK* zone names and UK00/UKNI home nodes.
# The PECD CSV already renamed them: UKOH* → GBOH*, UKNIOR1 → GBNIOR1.
# Handle both the zone name difference and the home node rename.
HOME_RENAME  = {"UK00": "GB00", "UKNI": "GBNI"}
ZONE_RENAME  = {"GB": "UK"}   # PECD CSV prefix → NODE.csv prefix

zone_to_home_raw = dict(zip(node_df["OFFSHORE_NODE"], node_df["HOME_NODE"]))

def lookup_bus(zone):
    home = zone_to_home_raw.get(zone)
    if home is None:
        for pecd_pfx, node_pfx in ZONE_RENAME.items():
            if zone.startswith(pecd_pfx):
                home = zone_to_home_raw.get(node_pfx + zone[len(pecd_pfx):])
                break
    return HOME_RENAME.get(home, home) if home else None

# Build final mapping: offshore zone → network bus code
offshore_raw = pd.read_csv(DATA_DIR / "pecd_data_Wind_Offshore_2030.csv", index_col=0, parse_dates=True)
zone_to_bus = {zone: lookup_bus(zone) for zone in offshore_raw.columns}



In [6]:
zone_to_bus

{'ALOR001': 'AL00',
 'BEIOH01': 'DKE1',
 'BEOH001': 'BE00',
 'BGOR001': 'BG00',
 'CYOR001': 'CY00',
 'DEOH001': 'DE00',
 'DEOH002': 'DE00',
 'DEOR001': 'DE00',
 'DKEOR01': 'DKE1',
 'DKWOH01': 'DKW1',
 'DKWOR01': 'DKW1',
 'EEOH001': 'EE00',
 'ESOH001': 'ES00',
 'ESOH002': 'ES00',
 'ESOH003': 'ES00',
 'FIOH001': 'FI00',
 'FROH001': 'FR00',
 'FROH002': 'FR00',
 'FROH003': 'FR00',
 'GROR001': 'GR00',
 'GROR002': 'GR03',
 'HROH001': 'HR00',
 'IEOH001': 'IE00',
 'ITCNOR1': 'ITCN',
 'ITCSOR1': 'ITCS',
 'ITN1OR1': 'ITN1',
 'ITS1OR1': 'ITS1',
 'ITSAOR1': 'ITSA',
 'ITSIOR1': 'ITSI',
 'LTOR001': 'LT00',
 'LVOH001': 'LV00',
 'MEOR001': 'ME00',
 'MTOR001': 'MT00',
 'NLOH001': 'NL00',
 'NLOR001': 'NL00',
 'NOMOH01': 'NOM1',
 'NONOH01': 'NON1',
 'NOSOH01': 'NOS0',
 'NOSOH02': 'NOS0',
 'NOSOR01': 'NOS0',
 'PLOH001': 'PL00',
 'PTOH001': 'PT00',
 'ROOR001': 'RO00',
 'SEOH001': 'SE02',
 'SEOH002': 'SE03',
 'SEOH003': 'SE04',
 'SEOR001': 'SE01',
 'UAOR001': 'UA02',
 'GBNIOR1': 'GBNI',
 'GBOH001': 'GB00',


In [7]:
print(set(zone_to_bus.values()))
print(len(set(zone_to_bus.values())))

{'FI00', 'NOS0', 'ITS1', 'ITN1', 'SE01', 'AL00', 'SE03', 'UA02', 'DKW1', 'DE00', 'HR00', 'PT00', 'ITCN', 'RO00', 'GR03', 'SE02', 'ITCS', 'LT00', 'ES00', 'GB00', 'ITSI', 'CY00', 'BG00', 'GR00', 'EE00', 'SE04', 'NL00', 'DKE1', 'FR00', 'NON1', 'NOM1', 'BE00', 'ITSA', 'GBNI', 'LV00', 'ME00', 'MT00', 'IE00', 'PL00'}
39


In [ ]:

# Validate against ACTUAL_NETWORK_BUSES
print(f"{'OFFSHORE_ZONE':<12}  {'HOME_NODE':<8}  STATUS")
print("-" * 42)
for zone, bus in sorted(zone_to_bus.items()):
    if bus in actual_buses:
        status = "✓"
    elif bus is None:
        status = "✗  no mapping in NODE.csv"
    else:
        status = f"✗  '{bus}' not in ACTUAL_NETWORK_BUSES"
    print(f"  {zone:<12}  →  {str(bus):<8}  {status}")

matched = [z for z, b in zone_to_bus.items() if b in actual_buses]
dropped = [z for z, b in zone_to_bus.items() if b not in actual_buses]
print(f"\nMapped OK : {len(matched)}")
print(f"Dropped   : {len(dropped)}  → {[f'{z}→{zone_to_bus[z]}' for z in dropped]}")

OFFSHORE_ZONE  HOME_NODE  STATUS
------------------------------------------
  ALOR001       →  AL00      ✓
  BEIOH01       →  DKE1      ✓
  BEOH001       →  BE00      ✓
  BGOR001       →  BG00      ✓
  CYOR001       →  CY00      ✓
  DEOH001       →  DE00      ✓
  DEOH002       →  DE00      ✓
  DEOR001       →  DE00      ✓
  DKEOR01       →  DKE1      ✓
  DKWOH01       →  DKW1      ✓
  DKWOR01       →  DKW1      ✓
  EEOH001       →  EE00      ✓
  ESOH001       →  ES00      ✓
  ESOH002       →  ES00      ✓
  ESOH003       →  ES00      ✓
  FIOH001       →  FI00      ✓
  FROH001       →  FR00      ✓
  FROH002       →  FR00      ✓
  FROH003       →  FR00      ✓
  GBNIOR1       →  GBNI      ✓
  GBOH001       →  GB00      ✓
  GBOH002       →  GB00      ✓
  GBOH003       →  GB00      ✓
  GBOH004       →  GB00      ✓
  GBOH005       →  GB00      ✓
  GBOH006       →  GB00      ✓
  GBOR001       →  GB00      ✓
  GROR001       →  GR00      ✓
  GROR002       →  GR03      ✓
  HROH001       →  HR00  

## Step 1 — Load Offshore Zone Capacities

Load `GENERATOR_ZONE_POTENTIAL.csv` and extract `EXISTING_MW` per offshore zone for **2030, National Trends**.

This gives the installed wind capacity (in MW) per TYNDP offshore zone — we will use these as weights when aggregating multiple zones into a single network bus.

In [9]:
zone_pot = pd.read_csv(DATA_DIR / "GENERATOR_ZONE_POTENTIAL.csv", sep=";")
zone_pot_30 = zone_pot[
    (zone_pot["YEAR"] == 2030) & (zone_pot["SCENARIO"] == "National Trends")
].copy()

# EXISTING_MW uses a comma as decimal separator in this CSV
zone_pot_30["EXISTING_MW"] = (
    zone_pot_30["EXISTING_MW"]
    .astype(str).str.replace(",", ".", regex=False).str.strip()
    .astype(float)
)

zone_capacity = zone_pot_30.set_index("OFFSHORE_NODE")["EXISTING_MW"]

print(f"Offshore zones loaded : {len(zone_capacity)}")
print(f"Zones with EXISTING_MW > 0 : {(zone_capacity > 0).sum()}")
print()
print(zone_capacity[zone_capacity > 0].sort_values(ascending=False).to_string())


Offshore zones loaded : 56
Zones with EXISTING_MW > 0 : 39

OFFSHORE_NODE
DEOH001    18868.00000
UKOR001    17533.61810
NLOR001    14300.00000
UKOH003    12722.98669
UKOH004    11012.77519
UKOH006    10398.09000
PLOH001    10004.00000
DEOR001     7548.75000
BEOH001     5760.00000
IEOH001     5025.20000
DEOH002     4104.55000
ITS1OR1     3800.00000
BEIOH01     3000.00000
DKWOR01     2722.40000
GROR001     2700.00000
DKEOR01     2528.20000
FROH001     2050.00000
PTOH001     2000.00000
NLOH001     2000.00000
FROH002     1740.00000
NOSOH01     1503.50000
NOSOR01     1500.00000
ESOH001     1400.00000
ITSIOR1     1400.00000
LTOR001     1400.00000
ITSAOR1     1400.00000
ROOR001     1000.00000
FIOH001     1000.00000
ESOH003      900.00000
ITCSOR1      900.00000
SEOH003      600.00000
ITN1OR1      600.00000
HROH001      510.00000
LVOH001      500.00000
UKNIOR1      500.00000
ESOH002      500.00000
ITCNOR1      400.00000
UKOH005      300.00000
FROH003       85.00000


## Step 2a — Assign a Capacity Weight to Each PECD Zone

Map each PECD offshore zone to its `EXISTING_MW` from Step 1.

- `GENERATOR_ZONE_POTENTIAL.csv` uses `UK*` zone names; the PECD CSV uses `GB*` — we handle the prefix swap.
- Zones with `EXISTING_MW = 0` (no installed capacity in 2030) are **excluded** — they should not contribute to the aggregated profile.

In [10]:
def lookup_capacity(zone):
    """Return EXISTING_MW for a PECD zone (GB↔UK prefix swap), or None if not found / zero."""
    w = zone_capacity.get(zone, None)
    if w is None:
        # PECD uses GB prefix, ZONE_POTENTIAL uses UK prefix
        alt = "UK" + zone[2:] if zone.startswith("GB") else zone
        w = zone_capacity.get(alt, None)
    if w is None or float(w) <= 0:
        return None
    return float(w)

# Keep only zones that:
#   1. map to a known network bus
#   2. have EXISTING_MW > 0 in 2030
valid_zones  = [
    z for z in offshore_raw.columns
    if zone_to_bus.get(z) in actual_buses and lookup_capacity(z) is not None
]
zone_weights = pd.Series({z: lookup_capacity(z) for z in valid_zones}, name="weight_MW")

print(f"Valid PECD zones (with capacity > 0) : {len(valid_zones)}")
print()
print(f"{'ZONE':<12}  {'→ BUS':<6}  {'WEIGHT (MW)':>12}")
print("-" * 38)
for z in valid_zones:
    print(f"  {z:<12}  {zone_to_bus[z]:<6}  {zone_weights[z]:>12.2f}")


Valid PECD zones (with capacity > 0) : 39

ZONE          → BUS    WEIGHT (MW)
--------------------------------------
  BEIOH01       DKE1         3000.00
  BEOH001       BE00         5760.00
  DEOH001       DE00        18868.00
  DEOH002       DE00         4104.55
  DEOR001       DE00         7548.75
  DKEOR01       DKE1         2528.20
  DKWOR01       DKW1         2722.40
  ESOH001       ES00         1400.00
  ESOH002       ES00          500.00
  ESOH003       ES00          900.00
  FIOH001       FI00         1000.00
  FROH001       FR00         2050.00
  FROH002       FR00         1740.00
  FROH003       FR00           85.00
  GROR001       GR00         2700.00
  HROH001       HR00          510.00
  IEOH001       IE00         5025.20
  ITCNOR1       ITCN          400.00
  ITCSOR1       ITCS          900.00
  ITN1OR1       ITN1          600.00
  ITS1OR1       ITS1         3800.00
  ITSAOR1       ITSA         1400.00
  ITSIOR1       ITSI         1400.00
  LTOR001       LT00         140

In [11]:
valid_zones

['BEIOH01',
 'BEOH001',
 'DEOH001',
 'DEOH002',
 'DEOR001',
 'DKEOR01',
 'DKWOR01',
 'ESOH001',
 'ESOH002',
 'ESOH003',
 'FIOH001',
 'FROH001',
 'FROH002',
 'FROH003',
 'GROR001',
 'HROH001',
 'IEOH001',
 'ITCNOR1',
 'ITCSOR1',
 'ITN1OR1',
 'ITS1OR1',
 'ITSAOR1',
 'ITSIOR1',
 'LTOR001',
 'LVOH001',
 'NLOH001',
 'NLOR001',
 'NOSOH01',
 'NOSOR01',
 'PLOH001',
 'PTOH001',
 'ROOR001',
 'SEOH003',
 'GBNIOR1',
 'GBOH003',
 'GBOH004',
 'GBOH005',
 'GBOH006',
 'GBOR001']

## Step 2b — Filter Out All-Zero PECD Columns

Before aggregating, verify which raw PECD offshore zones carry actual data (at least one non-zero hourly value) and compare that set against `valid_zones` from Step 2.


In [12]:
# Zones that have at least one non-zero value in the raw PECD data
nonzero_zones = [z for z in offshore_raw.columns if offshore_raw[z].any()]
allzero_zones = [z for z in offshore_raw.columns if not offshore_raw[z].any()]

valid_set   = set(valid_zones)
nonzero_set = set(nonzero_zones)

# Classification relative to valid_zones
in_both          = sorted(valid_set & nonzero_set)          # in valid_zones AND have data
in_valid_only    = sorted(valid_set - nonzero_set)          # in valid_zones BUT all zeros (bad)
in_nonzero_only  = sorted(nonzero_set - valid_set)          # have data BUT not in valid_zones
in_neither       = sorted(set(offshore_raw.columns) - valid_set - nonzero_set)  # all-zero AND not valid

print(f"Total PECD offshore zones  : {len(offshore_raw.columns)}")
print(f"  With at least one non-zero value : {len(nonzero_zones)}")
print(f"  All-zero columns                 : {len(allzero_zones)}")
print()
print(f"valid_zones (from Step 2)  : {len(valid_zones)}")
print()

print("─" * 60)
print(f"✓ In BOTH valid_zones AND non-zero ({len(in_both)}):")
for z in in_both:
    print(f"   {z:<12}  → bus: {zone_to_bus[z]}")

print()
print(f"⚠  In valid_zones but ALL-ZERO in PECD ({len(in_valid_only)}):")
if in_valid_only:
    for z in in_valid_only:
        print(f"   {z:<12}  → bus: {zone_to_bus[z]}")
else:
    print("   none")

print()
print(f"ℹ  Non-zero in PECD but NOT in valid_zones ({len(in_nonzero_only)}):")
if in_nonzero_only:
    for z in in_nonzero_only:
        bus = zone_to_bus.get(z, "—no mapping—")
        print(f"   {z:<12}  → bus: {bus}")
else:
    print("   none")

print()
print(f"—  All-zero AND not in valid_zones ({len(in_neither)}):")
if in_neither:
    for z in in_neither:
        bus = zone_to_bus.get(z, "—no mapping—")
        print(f"   {z:<12}  → bus: {bus}")
else:
    print("   none")



Total PECD offshore zones  : 56
  With at least one non-zero value : 46
  All-zero columns                 : 10

valid_zones (from Step 2)  : 39

────────────────────────────────────────────────────────────
✓ In BOTH valid_zones AND non-zero (38):
   BEIOH01       → bus: DKE1
   BEOH001       → bus: BE00
   DEOH001       → bus: DE00
   DEOH002       → bus: DE00
   DEOR001       → bus: DE00
   DKEOR01       → bus: DKE1
   DKWOR01       → bus: DKW1
   ESOH001       → bus: ES00
   ESOH002       → bus: ES00
   ESOH003       → bus: ES00
   FIOH001       → bus: FI00
   FROH001       → bus: FR00
   FROH002       → bus: FR00
   FROH003       → bus: FR00
   GBNIOR1       → bus: GBNI
   GBOH003       → bus: GB00
   GBOH004       → bus: GB00
   GBOH005       → bus: GB00
   GBOH006       → bus: GB00
   GBOR001       → bus: GB00
   GROR001       → bus: GR00
   IEOH001       → bus: IE00
   ITCNOR1       → bus: ITCN
   ITCSOR1       → bus: ITCS
   ITN1OR1       → bus: ITN1
   ITS1OR1       → bus: ITS

In [ ]:
# Offshore capacity aggregated by home bus
print()
print("─" * 60)
print("Offshore wind EXISTING_MW aggregated by bidding zone (home bus):")
print()
print(f"  {'BUS':<8}  {'ZONES':>5}  {'TOTAL EXISTING_MW':>18}  ZONE BREAKDOWN")
print("  " + "-" * 74)

from collections import defaultdict
bus_zones_map = defaultdict(list)
for z in valid_zones:
    bus_zones_map[zone_to_bus[z]].append(z)

for bus in sorted(bus_zones_map):
    zones = bus_zones_map[bus]
    total_mw = sum(zone_weights[z] for z in zones)
    breakdown = "  +  ".join(f"{z} ({zone_weights[z]:.0f} MW)" for z in sorted(zones))
    print(f"  {bus:<8}  {len(zones):>5}  {total_mw:>18.2f}  {breakdown}")



────────────────────────────────────────────────────────────
Offshore wind EXISTING_MW aggregated by bidding zone (home bus):

  BUS       ZONES   TOTAL EXISTING_MW  ZONE BREAKDOWN
  --------------------------------------------------------------------------
  BE00          1             5760.00  BEOH001 (5760 MW)
  DE00          3            30521.30  DEOH001 (18868 MW)  +  DEOH002 (4105 MW)  +  DEOR001 (7549 MW)
  DKE1          2             5528.20  BEIOH01 (3000 MW)  +  DKEOR01 (2528 MW)
  DKW1          1             2722.40  DKWOR01 (2722 MW)
  ES00          3             2800.00  ESOH001 (1400 MW)  +  ESOH002 (500 MW)  +  ESOH003 (900 MW)
  FI00          1             1000.00  FIOH001 (1000 MW)
  FR00          3             3875.00  FROH001 (2050 MW)  +  FROH002 (1740 MW)  +  FROH003 (85 MW)
  GB00          5            51967.47  GBOH003 (12723 MW)  +  GBOH004 (11013 MW)  +  GBOH005 (300 MW)  +  GBOH006 (10398 MW)  +  GBOR001 (17534 MW)
  GBNI          1              500.00  GBNI

In [14]:
import pandas as pd

# Build a tidy DataFrame: bus | total_existing_mw | zones (semicolon-separated)
rows = []
for bus in sorted(bus_zones_map):
    zones = bus_zones_map[bus]
    total_mw = sum(zone_weights[z] for z in zones)
    zone_breakdown = "; ".join(f"{z} ({zone_weights[z]:.0f} MW)" for z in sorted(zones))
    rows.append({
        "bus": bus,
        "total_existing_mw": round(total_mw, 2),
        "zones": zone_breakdown,
    })

offshore_capacity_df = pd.DataFrame(rows)

out_path = DATA_DIR / "offshore_wind_capacity_by_bus.csv"
offshore_capacity_df.to_csv(out_path, index=False)

print(f"Saved → {out_path}")
print(f"Shape : {offshore_capacity_df.shape}")
print()
print(offshore_capacity_df.to_string(index=False))


Saved → ../data/open-tyndp/offshore_wind_capacity_by_bus.csv
Shape : (26, 3)

 bus  total_existing_mw                                                                                            zones
BE00            5760.00                                                                                BEOH001 (5760 MW)
DE00           30521.30                                         DEOH001 (18868 MW); DEOH002 (4105 MW); DEOR001 (7549 MW)
DKE1            5528.20                                                             BEIOH01 (3000 MW); DKEOR01 (2528 MW)
DKW1            2722.40                                                                                DKWOR01 (2722 MW)
ES00            2800.00                                            ESOH001 (1400 MW); ESOH002 (500 MW); ESOH003 (900 MW)
FI00            1000.00                                                                                FIOH001 (1000 MW)
FR00            3875.00                                            FROH001 

## Step 3 — Capacity-Weighted Aggregation of PECD Profiles per Bus

For each network bus that has one or more offshore zones:

$$CF_{bus} = \frac{\sum_{z \in zones(bus)} CF_z \cdot w_z}{\sum_{z \in zones(bus)} w_z}$$

where $w_z$ is `EXISTING_MW` for zone $z$.  
Buses with a single zone are unaffected (trivially weighted).

In [15]:
buses_unique = sorted({zone_to_bus[z] for z in valid_zones})
wind_offshore = pd.DataFrame(index=offshore_raw.index, columns=buses_unique, dtype=float)

for bus in buses_unique:
    bus_zones = [z for z in valid_zones if zone_to_bus[z] == bus]
    w  = zone_weights[bus_zones].values          # numpy array of weights
    cf = offshore_raw[bus_zones].values          # (8760 × n_zones) array
    wind_offshore[bus] = (cf * w).sum(axis=1) / w.sum()

print(f"wind_offshore shape : {wind_offshore.shape}  ({wind_offshore.shape[0]} h × {wind_offshore.shape[1]} buses)")
print(f"CF range            : [{wind_offshore.min().min():.4f}, {wind_offshore.max().max():.4f}]")
assert wind_offshore.min().min() >= 0.0
assert wind_offshore.max().max() <= 1.0
print("All values in [0, 1] ✓")
print()

# Summary: how many zones feed each bus, and total weight
print(f"{'BUS':<6}  {'ZONES':>5}  {'TOTAL WEIGHT (MW)':>18}")
print("-" * 34)
for bus in buses_unique:
    bus_zones = [z for z in valid_zones if zone_to_bus[z] == bus]
    total_w = zone_weights[bus_zones].sum()
    print(f"  {bus:<6}  {len(bus_zones):>5}  {total_w:>18.2f}")


wind_offshore shape : (8760, 26)  (8760 h × 26 buses)
CF range            : [0.0000, 0.9500]
All values in [0, 1] ✓

BUS     ZONES   TOTAL WEIGHT (MW)
----------------------------------
  BE00        1             5760.00
  DE00        3            30521.30
  DKE1        2             5528.20
  DKW1        1             2722.40
  ES00        3             2800.00
  FI00        1             1000.00
  FR00        3             3875.00
  GB00        5            51967.47
  GBNI        1              500.00
  GR00        1             2700.00
  HR00        1              510.00
  IE00        1             5025.20
  ITCN        1              400.00
  ITCS        1              900.00
  ITN1        1              600.00
  ITS1        1             3800.00
  ITSA        1             1400.00
  ITSI        1             1400.00
  LT00        1             1400.00
  LV00        1              500.00
  NL00        2            16300.00
  NOS0        2             3003.50
  PL00        1       

## Step 3b — Patch HR00: inject profile from raw TYNDP source

`HROH001` in `pecd_data_Wind_Offshore_2030.csv` is all-zero (data gap in the pre-processed file),
but the raw TYNDP source `PECD_Wind_Offshore_2030_HR01_edition 2023.2.csv` has valid data.

We load the raw file, extract the **2009** climate-year column, align it to the model index, and
write it into `wind_offshore['HR00']`.


In [16]:
PECD_RAW_DIR = Path("../data/tyndp2024/PECD 2030")
CLIMATE_YEAR = "2009"   # column to extract from the raw file

hr_raw = pd.read_csv(
    PECD_RAW_DIR / "PECD_Wind_Offshore_2030_HR01_edition 2023.2.csv",
    header=10,
)

# Sanity check: column exists
assert CLIMATE_YEAR in hr_raw.columns, f"Column '{CLIMATE_YEAR}' not found in HR raw file"

# The raw file has 8760 rows (hours 1-24 × 365 days), same length as wind_offshore
hr_cf = hr_raw[CLIMATE_YEAR].values.astype(float)
assert len(hr_cf) == len(wind_offshore), (
    f"Length mismatch: raw HR={len(hr_cf)}, wind_offshore={len(wind_offshore)}"
)

# Add HR00 column (or overwrite if already present from Step 3)
wind_offshore["HR00"] = hr_cf

print(f"HR00 patched from raw TYNDP file (climate year {CLIMATE_YEAR})")
print(f"  min={wind_offshore['HR00'].min():.4f}  "
      f"max={wind_offshore['HR00'].max():.4f}  "
      f"mean={wind_offshore['HR00'].mean():.4f}  "
      f"non-zero={(wind_offshore['HR00'] != 0).sum()} / {len(wind_offshore)}")
assert wind_offshore["HR00"].between(0, 1).all(), "HR00 values outside [0, 1]!"
print("All values in [0, 1] ✓")


HR00 patched from raw TYNDP file (climate year 2009)
  min=0.0000  max=0.9500  mean=0.2628  non-zero=8335 / 8760
All values in [0, 1] ✓


In [17]:
out_path = DATA_DIR / "pecd_data_Wind_Offshore_2030_mapped.csv"
wind_offshore.to_csv(out_path)
print(f"Saved → {out_path}")
print(f"Shape : {wind_offshore.shape}")


Saved → ../data/open-tyndp/pecd_data_Wind_Offshore_2030_mapped.csv
Shape : (8760, 26)


In [18]:
wind_offshore.columns

Index(['BE00', 'DE00', 'DKE1', 'DKW1', 'ES00', 'FI00', 'FR00', 'GB00', 'GBNI',
       'GR00', 'HR00', 'IE00', 'ITCN', 'ITCS', 'ITN1', 'ITS1', 'ITSA', 'ITSI',
       'LT00', 'LV00', 'NL00', 'NOS0', 'PL00', 'PT00', 'RO00', 'SE04'],
      dtype='object')

In [ ]:
# Step-by-step example: DE00, first timestamp
bus = "DE00"
t   = offshore_raw.index[0]

de_zones = [z for z in valid_zones if zone_to_bus[z] == bus]
w_de     = zone_weights[de_zones]
cf_de    = offshore_raw.loc[t, de_zones]

print(f"Bus : {bus}")
print(f"Time: {t}")
print()
print(f"  {'ZONE':<12}  {'CF':>8}  {'WEIGHT (MW)':>12}  {'CF × weight':>12}")
print("  " + "-" * 52)
for z in de_zones:
    print(f"  {z:<12}  {cf_de[z]:>8.4f}  {w_de[z]:>12.1f}  {cf_de[z]*w_de[z]:>12.4f}")

print()
print(f"  Sum of (CF × weight) : {(cf_de * w_de).sum():.4f}")
print(f"  Sum of weights       : {w_de.sum():.1f} MW")
print(f"  ─────────────────────────────────────────")
print(f"  Weighted avg CF      : {(cf_de * w_de).sum() / w_de.sum():.6f}")
print()
print(f"  wind_offshore['DE00'] at t=0 : {wind_offshore.loc[t, bus]:.6f}  ← should match")


Bus : DE00
Time: 2009-01-01 00:00:00

  ZONE                CF   WEIGHT (MW)   CF × weight
  ----------------------------------------------------
  DEOH001         0.2679       18868.0     5055.1306
  DEOH002         0.1408        4104.6      577.7630
  DEOR001         0.2106        7548.8     1589.6153

  Sum of (CF × weight) : 7222.5090
  Sum of weights       : 30521.3 MW
  ─────────────────────────────────────────
  Weighted avg CF      : 0.236638

  wind_offshore['DE00'] at t=0 : 0.236638  ← should match


In [20]:
display(wind_offshore)

,BE00,DE00,DKE1,DKW1,ES00,FI00,FR00,GB00,GBNI,GR00,...,ITSA,ITSI,LT00,LV00,NL00,NOS0,PL00,PT00,RO00,SE04
2009-01-01 00:00:00,0.006736,0.236638,0.161902,0.594808,0.605975,0.922741,0.482505,0.157602,0.110723,0.733920,...,0.621261,0.735950,0.194373,0.225230,0.056180,0.678014,0.217605,0.782285,0.057370,0.187278
2009-01-01 01:00:00,0.004962,0.262659,0.184009,0.578846,0.582455,0.918446,0.440913,0.159458,0.138486,0.686058,...,0.571401,0.655615,0.202553,0.306011,0.053232,0.760109,0.242895,0.692790,0.053415,0.231050
2009-01-01 02:00:00,0.004598,0.293127,0.217224,0.552811,0.541922,0.921929,0.430248,0.157594,0.150683,0.650440,...,0.508567,0.612352,0.209820,0.404716,0.044081,0.820321,0.252767,0.644933,0.050385,0.257646
2009-01-01 03:00:00,0.006754,0.305920,0.246343,0.530000,0.480038,0.922233,0.430664,0.160796,0.152147,0.637414,...,0.435786,0.619447,0.229257,0.529568,0.040384,0.842138,0.270651,0.692265,0.043410,0.262208
2009-01-01 04:00:00,0.007690,0.316546,0.234236,0.504549,0.424258,0.904696,0.451714,0.163851,0.146125,0.645021,...,0.359448,0.661733,0.302384,0.674396,0.048526,0.786662,0.315569,0.788860,0.031264,0.307086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2009-12-31 19:00:00,0.829624,0.550622,0.135394,0.208676,0.625105,0.153154,0.931139,0.612081,0.402355,0.787948,...,0.934638,0.356027,0.097891,0.143573,0.920319,0.011543,0.049024,0.949796,0.869639,0.047574
2009-12-31 20:00:00,0.855016,0.549186,0.130435,0.186524,0.636313,0.156964,0.935978,0.603088,0.338470,0.776272,...,0.947384,0.398408,0.082691,0.110953,0.905979,0.022341,0.057813,0.949839,0.768592,0.064319
2009-12-31 21:00:00,0.872280,0.543486,0.129784,0.164355,0.636004,0.156967,0.937378,0.593985,0.292478,0.763820,...,0.948565,0.532429,0.062257,0.088181,0.891871,0.023540,0.054889,0.949948,0.563696,0.073459
2009-12-31 22:00:00,0.841937,0.590109,0.166103,0.171398,0.598168,0.154750,0.920012,0.563335,0.211876,0.754475,...,0.944334,0.815994,0.028161,0.068421,0.855862,0.001612,0.043977,0.950000,0.349828,0.089675


## Conclusion

The aggregated `wind_offshore` DataFrame:
- Uses **bus codes** matching `buses.csv`
- Values are **capacity factors** in `[0, 1]`
- Zones are aggregated using a **capacity-weighted average** based on `EXISTING_MW` from `GENERATOR_ZONE_POTENTIAL.csv` (2030, National Trends)
- Zones with no recorded existing capacity fall back to weight = 1.0

The resulting CSV (`pecd_data_Wind_Offshore_2030_mapped.csv`) is used in `test.ipynb` as `p_max_pu` for offshore wind generators.